In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import shutil

def remove_specular_highlights(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 0, 240), (180, 40, 255))

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    inpainted = cv2.inpaint(img, mask, 5, cv2.INPAINT_TELEA)

    return inpainted, mask   

def kger_preprocessing(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_clahe = clahe.apply(gray)

    blurred = cv2.GaussianBlur(gray_clahe, (31, 31), 0)

    circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=80,
        param1=30,   
        param2=20,   
        minRadius=int(0.20 * img.shape[1]),
        maxRadius=int(0.60 * img.shape[1])
    )

    if circles is None:
        print("No circle detected. Returning CLAHE image.")
        return gray_clahe

    circles = np.uint16(np.around(circles))[0, 0]
    x, y, r = circles

    mask = np.zeros_like(gray_clahe)
    cv2.circle(mask, (x, y), int(r * 1.1), 255, -1)
    cropped_circle = cv2.bitwise_and(gray_clahe, gray_clahe, mask=mask)

    th = cv2.threshold(cropped_circle, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        print("No contour found. Returning circular crop.")
        return cropped_circle

    c = max(contours, key=cv2.contourArea)
    x2, y2, w2, h2 = cv2.boundingRect(c)

    final_crop = cropped_circle[y2:y2+h2, x2:x2+w2]
    return final_crop

In [2]:
def process_split(split_name):
    in_dir = f"{ROOT}/{split_name}/images"
    out_dir = f"{OUT_ROOT}/{split_name}/images"
    os.makedirs(out_dir, exist_ok=True)

    print(f"Processing: {split_name}")

    files = os.listdir(in_dir)

    for fname in tqdm(files):
        if not fname.lower().endswith((".jpg", ".png", ".jpeg", ".bmp")):
            print("Skipping non-image:", fname)
            continue

        path = f"{in_dir}/{fname}"
        img = cv2.imread(path)

        if img is None:
            print("Cannot load image:", path)
            continue

        img, mask = remove_specular_highlights(img)
        processed = kger_preprocessing(img)

        if processed is None:
            print("Processing failed for:", fname)
            continue

        cv2.imwrite(f"{out_dir}/{fname}", processed)

def copy_metadata(split_name):
    meta_in = f"{ROOT}/{split_name}/metadata_{split_name}.jsonl"
    meta_out = f"{OUT_ROOT}/{split_name}/metadata_{split_name}.jsonl"

    if os.path.exists(meta_in):
        os.makedirs(f"{OUT_ROOT}/{split_name}", exist_ok=True)
        shutil.copy(meta_in, meta_out)
        print(f"Copied metadata for {split_name}")

In [3]:
ROOT = "/kaggle/input/sumotosima"
OUT_ROOT = "/kaggle/working/sumotoshima"

os.makedirs(OUT_ROOT, exist_ok=True)

for split in ["train", "val", "test"]:
    process_split(split)
    copy_metadata(split)

print("\nDONE! All images processed successfully.")

Processing: train


100%|██████████| 400/400 [00:41<00:00,  9.60it/s]


Copied metadata for train
Processing: val


100%|██████████| 50/50 [00:04<00:00, 10.87it/s]


Copied metadata for val
Processing: test


100%|██████████| 50/50 [00:05<00:00,  9.05it/s]

Copied metadata for test

DONE! All images processed successfully.


In [4]:
import shutil

zip_path = "/kaggle/working/sumotoshima_processed.zip"
folder_path = "/kaggle/working/sumotoshima"

shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)

print("ZIP created at:", zip_path)

ZIP created at: /kaggle/working/sumotoshima_processed.zip
